In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, Callback
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, BatchNormalization, LeakyReLU, MaxPooling2D, Flatten, Dense, Dropout, Add, Input
from tensorflow.keras.regularizers import l2

DeepNav

In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ReduceLROnPlateau

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load training and validation data -----------
train_df = pd.read_csv('./csbf3nvm/NVM23_train_split.txt', delimiter=' ',
                       names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

val_df = pd.read_csv('./csbf3nvm/NVM23_val_split.txt', delimiter=' ',
                     names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build CNN model (no pretrained) -----------
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(1024, activation='relu'),
    Dropout(0.5),

    Dense(512, activation='relu'),
    Dropout(0.5),

    Dense(7)  # Output: [X, Y, Z, W, P, Q, R]
])
# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss=custom_loss,
              metrics=['mae'])

# ----------- 5. Custom callback (updated to log both train and val loss) -----------
class TrainValLossLogger(Callback):
    def __init__(self, train_gen, val_gen, output_file='loss_CNN_FIX.csv'):
        super().__init__()
        self.train_gen = train_gen
        self.val_gen = val_gen
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        def compute_losses(gen):
            x, y_true = next(iter(gen))
            y_pred = self.model.predict(x)
            x_pred, q_pred = y_pred[:, :3], y_pred[:, 3:]
            x_true, q_true = y_true[:, :3], y_true[:, 3:]
            q_true_normed = q_true / (np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7)
            pos_loss = np.mean(np.square(x_pred - x_true))
            quat_loss = np.mean(np.square(q_pred - q_true_normed))
            total_loss = pos_loss + 400. * quat_loss
            return total_loss, pos_loss, quat_loss

        train_total, train_pos, train_quat = compute_losses(self.train_gen)
        val_total, val_pos, val_quat = compute_losses(self.val_gen)

        self.logs.append({
            'Epoch': epoch + 1,
            'Train_loss': train_total,
            'Train_pos_loss': train_pos,
            'Train_quat_loss': train_quat,
            'Val_loss': val_total,
            'Val_pos_loss': val_pos,
            'Val_quat_loss': val_quat
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: train_loss={train_total:.4f}, val_loss={val_total:.4f}")

# ----------- 6. Callbacks -----------
loss_logger = TrainValLossLogger(train_generator, val_generator)
checkpoint = ModelCheckpoint('best_CNN_FIX_model.h5', monitor='val_loss',save_best_only=True,mode='min',verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-8, verbose=1)

# ----------- 7. Train with validation -----------
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=200,
    callbacks=[checkpoint,loss_logger,reduce_lr]
)

Mobile Net

In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
train_df = pd.read_csv('./csbf3nvm/NVM23_train_split.txt', delimiter=' ',
                       names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

val_df = pd.read_csv('./csbf3nvm/NVM23_val_split.txt', delimiter=' ',
                     names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=False
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build model with MobileNetV3 -----------
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = True  # Set to True later for fine-tuning

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=True)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(7)(x)  # Output: [X, Y, Z, W, P, Q, R]

model = Model(inputs, outputs)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
# ----------- 5. Custom callback (updated to log both train and val loss) -----------
class TrainValLossLogger(Callback):
    def __init__(self, train_gen, val_gen, output_file='loss_mobile.csv'):
        super().__init__()
        self.train_gen = train_gen
        self.val_gen = val_gen
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        def compute_losses(gen):
            x, y_true = next(iter(gen))
            y_pred = self.model.predict(x)
            x_pred, q_pred = y_pred[:, :3], y_pred[:, 3:]
            x_true, q_true = y_true[:, :3], y_true[:, 3:]
            q_true_normed = q_true / (np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7)
            pos_loss = np.mean(np.square(x_pred - x_true))
            quat_loss = np.mean(np.square(q_pred - q_true_normed))
            total_loss = pos_loss + 400. * quat_loss
            return total_loss, pos_loss, quat_loss

        train_total, train_pos, train_quat = compute_losses(self.train_gen)
        val_total, val_pos, val_quat = compute_losses(self.val_gen)

        self.logs.append({
            'Epoch': epoch + 1,
            'Train_loss': train_total,
            'Train_pos_loss': train_pos,
            'Train_quat_loss': train_quat,
            'Val_loss': val_total,
            'Val_pos_loss': val_pos,
            'Val_quat_loss': val_quat
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: train_loss={train_total:.4f}, val_loss={val_total:.4f}")

# ----------- 6. Callbacks -----------
loss_logger = TrainValLossLogger(train_generator, val_generator)
# ----------- 7. Train with validation -----------
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=200,
    callbacks=[loss_logger]
)


Efficient Net

In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
train_df = pd.read_csv('./csbf3nvm/NVM23_train_split.txt', delimiter=' ',
                       names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

val_df = pd.read_csv('./csbf3nvm/NVM23_val_split.txt', delimiter=' ',
                     names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=False
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build EfficientNetV2B0 model -----------
base_model = EfficientNetV2B0(include_top=False, input_shape=(224, 224, 3), weights='imagenet')

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(7)(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
# ----------- 5. Custom callback (updated to log both train and val loss) -----------
class TrainValLossLogger(Callback):
    def __init__(self, train_gen, val_gen, output_file='loss_EfficientNetV2B0.csv'):
        super().__init__()
        self.train_gen = train_gen
        self.val_gen = val_gen
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        def compute_losses(gen):
            x, y_true = next(iter(gen))
            y_pred = self.model.predict(x)
            x_pred, q_pred = y_pred[:, :3], y_pred[:, 3:]
            x_true, q_true = y_true[:, :3], y_true[:, 3:]
            q_true_normed = q_true / (np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7)
            pos_loss = np.mean(np.square(x_pred - x_true))
            quat_loss = np.mean(np.square(q_pred - q_true_normed))
            total_loss = pos_loss + 400. * quat_loss
            return total_loss, pos_loss, quat_loss

        train_total, train_pos, train_quat = compute_losses(self.train_gen)
        val_total, val_pos, val_quat = compute_losses(self.val_gen)

        self.logs.append({
            'Epoch': epoch + 1,
            'Train_loss': train_total,
            'Train_pos_loss': train_pos,
            'Train_quat_loss': train_quat,
            'Val_loss': val_total,
            'Val_pos_loss': val_pos,
            'Val_quat_loss': val_quat
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: train_loss={train_total:.4f}, val_loss={val_total:.4f}")

# ----------- 6. Callbacks (Remove checkpoint) -----------
loss_logger = TrainValLossLogger(train_generator, val_generator)

# ----------- 7. Train with validation -----------
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=200,
    callbacks=[loss_logger]
)

# ----------- 8. Save final model (last epoch) -----------
model.save('final_model_EfficientNetV2B0.h5')

Inception

In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Flatten
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
train_df = pd.read_csv('./csbf3nvm/NVM23_train_split.txt', delimiter=' ',
                       names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

val_df = pd.read_csv('./csbf3nvm/NVM23_val_split.txt', delimiter=' ',
                     names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=False
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build InceptionV3 model -----------
base_model = InceptionV3(include_top=False, input_shape=(224, 224, 3), weights='imagenet')

x = base_model.output
x = Flatten()(x) 
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(7)(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
class TrainValLossLogger(Callback):
    def __init__(self, train_gen, val_gen, output_file='loss_InceptionV3_.csv'):
        super().__init__()
        self.train_gen = train_gen
        self.val_gen = val_gen
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        def compute_losses(gen):
            x, y_true = next(iter(gen))
            y_pred = self.model.predict(x)
            x_pred, q_pred = y_pred[:, :3], y_pred[:, 3:]
            x_true, q_true = y_true[:, :3], y_true[:, 3:]
            q_true_normed = q_true / (np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7)
            pos_loss = np.mean(np.square(x_pred - x_true))
            quat_loss = np.mean(np.square(q_pred - q_true_normed))
            total_loss = pos_loss + 400. * quat_loss
            return total_loss, pos_loss, quat_loss

        train_total, train_pos, train_quat = compute_losses(self.train_gen)
        val_total, val_pos, val_quat = compute_losses(self.val_gen)

        self.logs.append({
            'Epoch': epoch + 1,
            'Train_loss': train_total,
            'Train_pos_loss': train_pos,
            'Train_quat_loss': train_quat,
            'Val_loss': val_total,
            'Val_pos_loss': val_pos,
            'Val_quat_loss': val_quat
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: train_loss={train_total:.4f}, val_loss={val_total:.4f}")

# ----------- 6. Callbacks -----------
loss_logger = TrainValLossLogger(train_generator, val_generator)

# ----------- 7. Train with validation -----------
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=200,
    callbacks=[loss_logger]
)

model.save('final_model_InceptionV3_.h5')